In [1]:
!nvidia-smi


Wed Jan 14 13:03:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!apt-get install -y poppler-utils > /dev/null

In [3]:
!pip install transformers accelerate pillow pdf2image pytorch-lightning --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.5/849.5 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 62.9 MB/s eta 0:00:00


In [11]:
!apt-get -y install poppler-utils tesseract-ocr tesseract-ocr-swe > /dev/null
!pip -q install transformers accelerate pdf2image pytesseract


In [12]:
import torch
from transformers import AutoProcessor, VisionEncoderDecoderModel

processor = AutoProcessor.from_pretrained("facebook/nougat-small")
model = VisionEncoderDecoderModel.from_pretrained("facebook/nougat-small")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print("Device:", device)


Device: cuda


In [4]:
# 3. Import libraries
# ============================
from transformers import AutoProcessor, AutoTokenizer, VisionEncoderDecoderModel
from pdf2image import convert_from_path
import torch

In [ ]:
import os

def load_pdfs(folder: str = "data/benchmark") -> dict:
    try:
        from google.colab import files as colab_files

        print("Upload PDF files:")
        uploaded = colab_files.upload()
        pdfs = {
            fname: content
            for fname, content in uploaded.items()
            if fname.endswith(".pdf")
        }

    except ImportError:
        pdfs = {}
        for fname in os.listdir(folder):
            if not fname.endswith(".pdf"):
                continue
            with open(os.path.join(folder, fname), "rb") as f:
                pdfs[fname] = f.read()

    return pdfs 

pdfs = load_pdfs()
 


Saving aa602a2b-0cf0-5d8f-8050-4e73aa4e3d6c_2.pdf to aa602a2b-0cf0-5d8f-8050-4e73aa4e3d6c_2.pdf


In [6]:
pages = convert_from_path(pdf_path, dpi=150)
print("Pages loaded:", len(pages))


Pages loaded: 68


In [14]:
!pip -q install PyPDF2


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.6 MB/s eta 0:00:00


In [15]:
#page loaded and output
import os, gc
from pdf2image import convert_from_path
import pytesseract
from google.colab import files

# Use your uploaded pdf_path variable (important!)
print("Using PDF:", pdf_path)

# Count pages without loading all into RAM
from PyPDF2 import PdfReader
reader = PdfReader(open(pdf_path, "rb"))
total_pages = len(reader.pages)
print("Total pages:", total_pages)

def looks_like_hallucination(text: str) -> bool:
    t = (text or "").lower()
    # common Nougat hallucination patterns on non-scientific docs
    bad_signals = ["references", "appendix", "abstract", r"\begin{tabular}", "phys. rev", "arxiv"]
    if len(text.strip()) < 200:
        return True
    return any(s in t for s in bad_signals)

parts = []

for page_num in range(1, total_pages + 1):
    print(f"[Chandra/Nougat] page {page_num}/{total_pages}")

    page = convert_from_path(pdf_path, dpi=200, first_page=page_num, last_page=page_num)[0]

    #  Nougat attempt
    pixel_values = processor(images=page, return_tensors="pt").pixel_values.to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            pixel_values,
            max_new_tokens=1024,
            do_sample=False,
            num_beams=1
        )

    md = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    #  Fallback to Tesseract if Nougat seems wrong
    if looks_like_hallucination(md):
        ocr_text = pytesseract.image_to_string(page, lang="swe").strip()
        page_text = ocr_text
    else:
        page_text = md

    if page_text:
        parts.append(page_text)

    del page, pixel_values, generated_ids
    gc.collect()

full_text = "\n\n".join(parts).strip()
print("DONE. Characters:", len(full_text))
print(full_text[:600])

# Save output
output_filename = "chandra_output.md"
with open(output_filename, "w", encoding="utf-8") as f:
    f.write(full_text)

files.download(output_filename)
print(" Downloaded:", output_filename)


Using PDF: aa602a2b-0cf0-5d8f-8050-4e73aa4e3d6c_2.pdf
Total pages: 68
[Chandra/Nougat] page 1/68
[Chandra/Nougat] page 2/68
[Chandra/Nougat] page 3/68
[Chandra/Nougat] page 4/68
[Chandra/Nougat] page 5/68
[Chandra/Nougat] page 6/68
[Chandra/Nougat] page 7/68
[Chandra/Nougat] page 8/68
[Chandra/Nougat] page 9/68
[Chandra/Nougat] page 10/68
[Chandra/Nougat] page 11/68
[Chandra/Nougat] page 12/68
[Chandra/Nougat] page 13/68
[Chandra/Nougat] page 14/68
[Chandra/Nougat] page 15/68
[Chandra/Nougat] page 16/68
[Chandra/Nougat] page 17/68
[Chandra/Nougat] page 18/68
[Chandra/Nougat] page 19/68
[Chandra/Nougat] page 20/68
[Chandra/Nougat] page 21/68
[Chandra/Nougat] page 22/68
[Chandra/Nougat] page 23/68
[Chandra/Nougat] page 24/68
[Chandra/Nougat] page 25/68
[Chandra/Nougat] page 26/68
[Chandra/Nougat] page 27/68
[Chandra/Nougat] page 28/68
[Chandra/Nougat] page 29/68
[Chandra/Nougat] page 30/68
[Chandra/Nougat] page 31/68
[Chandra/Nougat] page 32/68
[Chandra/Nougat] page 33/68
[Chandra/Nougat

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 Downloaded: chandra_output.md


In [ ]:
files.download("chandra_output.md")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
#Goldstandard text
import PyPDF2

pdf_path = list(uploaded.keys())[0]
text = ""

with open(pdf_path, "rb") as f:
    reader = PyPDF2.PdfReader(f)
    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n\n"

with open("gold_standard.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("gold_standard.txt created!")
print(text[:1000])  # preview first 1000 chars


gold_standard.txt created!
Årsredovisning 
2022


Text
Innehåll
ÖVERSIKT
Vår historia  ....................................................................................................... 6
Året i korthet  .................................................................................................. 8
Sammanfattande nyckeltal & KPI:er  ................................................................ 10
VD har ordet  .................................................................................................... 12
STRATEGI
Marknad och Trender  ...................................................................................... 18
Strategisk Modell ............................................................................................. 20 
 
Strategisk Riktning  ........................................................................................... 22
Kundbas  ..........................................................................................................

In [ ]:
!pip install jiwer --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 56.7 MB/s eta 0:00:00


In [ ]:
from jiwer import wer, cer

# Load Chandra OCR output
with open("chandra_output.md", "r", encoding="utf-8") as f:
    chandra = f.read()

# Load Gold Standard
with open("gold_standard.txt", "r", encoding="utf-8") as f:
    gold = f.read()

# Compute metrics
chandra_wer = wer(gold, chandra)
chandra_cer = cer(gold, chandra)

print("Chandra WER:", chandra_wer)
print("Chandra CER:", chandra_cer)


Chandra WER: 0.9982189503680836
Chandra CER: 0.9882926205648651


In [ ]:
with open("gold_standard.txt", "r", encoding="utf-8") as f:
    print(f.read()[:500])


Årsredovisning 
2022


Text
Innehåll
ÖVERSIKT
Vår historia  ....................................................................................................... 6
Året i korthet  .................................................................................................. 8
Sammanfattande nyckeltal & KPI:er  ................................................................ 10
VD har ordet  ....................................................................................................


In [ ]:
with open("chandra_output.md", "r", encoding="utf-8") as f:
    print(f.read()[:500])


* [15] J. M. C. Collins, _Phys. Rev

## Part II FlexOutes

The flexible or graphical resource embedding is a central problem in

.

.

\begin{tabular}{c} \begin{tabular}

.

\begin{tabular}{c}  \\ \end{tab

.

.

## 6. Introduction

The study of the impact of the impact of

.

###### Abstract

We present a new new method for the determination of the

## References

* [1] A. B. K. Jain, Phys.

.

\begin{tabular}{c} \begin{tabular}

Strategisk Modell

Var strategy bygger pla fya komponenteter

Str


In [ ]:
#Convert chandra Markdown → Plain Text (no images)

import re

def md_to_text(md):
    # Remove Markdown images
    md = re.sub(r'!\[.*?\]\(.*?\)', '', md)
    # Remove Markdown headings
    md = re.sub(r'#.*', '', md)
    # Remove extra spaces
    md = re.sub(r'\n+', '\n', md)
    return md.strip()

with open("chandra_output.md", "r", encoding="utf-8") as f:
    raw_md = f.read()

mistral_text = md_to_text(raw_md)

with open("chandra_text.txt", "w", encoding="utf-8") as f:
    f.write(mistral_text)

print("Clean text saved as chandra_text.txt")


Clean text saved as chandra_text.txt


In [ ]:
print("Length of Gold Standard:", len(gold))
print("Length of Chandra output:", len(chandra))


Length of Gold Standard: 203379
Length of Chandra output: 2580


In [ ]:
#Visualize Chandra OCR errors by HTML diff

from difflib import HtmlDiff

diff = HtmlDiff().make_file(
    gold.splitlines(),
    chandra.splitlines(),
    fromdesc='Gold Standard',
    todesc='Chandra OCR Output'
)

with open("chandra_diff.html", "w", encoding="utf-8") as f:
    f.write(diff)

print("Chandra diff visualization saved to chandra_diff.html")


Chandra diff visualization saved to chandra_diff.html


In [ ]:
from google.colab import files
files.download("chandra_diff.html")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>